# AFM Quickstart: Probabilistic Time Series Forecasting

This notebook shows how to use Autoregressive Flow Matching (AFM) to train a model and generate probabilistic predictions on the Lorenz system.

In [ ]:
import jax
import jax.numpy as jnp
import jax.random as jrandom
import flax.nnx as nnx
import numpy as np
import matplotlib.pyplot as plt

from afm import AutoregressiveFlowMatching, Config, create_system
from afm.training.data import prepare_trajectories, create_infinite_dataloaders
from afm.training.trainers import AFMTrainer

## 1. Setup: Create a Dynamical System and Generate Data

We'll use the stochastic Lorenz system (a 3D chaotic attractor with additive noise).

In [ ]:
system = create_system("lorenz", stochastic=True)

config = Config(
    system_name="lorenz",
    system_stochastic=True,
    system_num_timepoints=100,
    system_t0=0.0,
    system_t1=2.0,
    context_length=50,
    data_n_train_test_samples=512,
    data_batch_size=64,
    steps_per_epoch=100,
    training_num_epochs=50,
)

key = jrandom.PRNGKey(0)
train_traj, test_traj, ts = prepare_trajectories(system, config, key)

print(f"Train trajectories: {train_traj.shape}")
print(f"Test trajectories:  {test_traj.shape}")
print(f"Time steps:         {ts.shape}")

Let's visualize a few training trajectories.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
labels = ["$x$", "$y$", "$z$"]
ts_np = np.array(ts)

for i in range(min(8, train_traj.shape[0])):
    traj = np.array(train_traj[i])
    for d in range(3):
        axes[d].plot(ts_np, traj[:, d], alpha=0.4, linewidth=0.8)

for d in range(3):
    axes[d].set_ylabel(labels[d])
    axes[d].grid(True, alpha=0.3)

axes[-1].set_xlabel("Time")
axes[0].set_title("Sample Lorenz Trajectories (Stochastic)")
plt.tight_layout()
plt.show()

## 2. Create Dataloaders

The infinite dataloader dynamically samples (context, next-state) pairs from trajectories for efficient training.

In [ ]:
(afm_train, afm_test), _ = create_infinite_dataloaders(
    train_traj, test_traj, ts, config
)

# Inspect one batch
for history, covariates, prediction in afm_train:
    print(f"History:    {history.shape}")     # (batch, context_len, 3)
    print(f"Covariates: {covariates.shape}")  # (batch, context_len, 1)
    print(f"Prediction: {prediction.shape}")  # (batch, 1, 3) - single next step
    break

## 3. Build and Train the AFM Model

In [ ]:
model = AutoregressiveFlowMatching(
    dim_x=system.data_dim,
    dim_c=config.model_covariates_dim,
    dim_h=config.model_hidden_dim,
    dim_t=config.model_time_dim,
    dim_hidden=config.model_hidden_dim,
    hidden_blocks=config.model_hidden_blocks,
    n_rnn_layers=config.model_n_rnn_layers,
    rnn_hidden=config.model_rnn_hidden,
    rngs=nnx.Rngs(0),
)

trainer = AFMTrainer(model, learning_rate=config.training_learning_rate)
train_losses, test_losses = trainer.train(
    afm_train, afm_test, num_epochs=50, use_wandb=False
)

In [ ]:
plt.figure(figsize=(8, 4))
plt.semilogy(train_losses, label="Train")
plt.semilogy(test_losses, label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Progress")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Generate Probabilistic Predictions

Given a history window, AFM autoregressively generates multiple stochastic forecast samples by solving an ODE from random noise to the predicted state at each step.

In [ ]:
# Pick a test trajectory
test_idx = 0
context_len = config.context_length
pred_len = 30

history = test_traj[test_idx, :context_len]       # (context_len, 3)
covariates = ts[:context_len, None]                 # (context_len, 1)

# Generate 50 probabilistic forecast samples
samples = model.sample(
    jrandom.PRNGKey(42),
    history[None],       # (1, context_len, 3)
    covariates[None],    # (1, context_len, 1)
    nfe=20,
    n_steps_pred=pred_len,
    n_samples=50,
)

print(f"Samples shape: {samples.shape}")  # (n_samples, 1, pred_len, 3)

## 5. Visualize Predictions with Uncertainty

In [ ]:
samples_np = np.array(samples[:, 0, :, :])  # (50, pred_len, 3)
mean = samples_np.mean(axis=0)
lower = np.percentile(samples_np, 2.5, axis=0)
upper = np.percentile(samples_np, 97.5, axis=0)

ground_truth = np.array(test_traj[test_idx])
history_np = np.array(history)

fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
labels = ["$x$", "$y$", "$z$"]

hist_ts = ts_np[:context_len]
pred_ts = ts_np[context_len : context_len + pred_len]

for d in range(3):
    ax = axes[d]

    # History
    ax.plot(hist_ts, history_np[:, d], "k-", linewidth=1.2, label="History")

    # Ground truth future
    ax.plot(pred_ts, ground_truth[context_len : context_len + pred_len, d],
            "k--", linewidth=1.0, alpha=0.6, label="Ground truth")

    # AFM mean prediction
    ax.plot(pred_ts, mean[:, d], "C0-", linewidth=1.2, label="AFM mean")

    # 95% confidence interval
    ax.fill_between(pred_ts, lower[:, d], upper[:, d],
                    color="C0", alpha=0.2, label="95% CI")

    # A few individual samples
    for s in range(5):
        ax.plot(pred_ts, samples_np[s, :, d], "C0-", alpha=0.1, linewidth=0.5)

    ax.set_ylabel(labels[d])
    ax.grid(True, alpha=0.3)
    if d == 0:
        ax.legend(fontsize=8, loc="upper right")

    # Shade regions
    ax.axvspan(hist_ts[0], hist_ts[-1], color="gray", alpha=0.06)
    ax.axvspan(pred_ts[0], pred_ts[-1], color="blue", alpha=0.03)

axes[-1].set_xlabel("Time")
axes[0].set_title("AFM Probabilistic Forecast — Lorenz System")
plt.tight_layout()
plt.savefig("afm_lorenz_forecast.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Try a Different System

The same workflow applies to any of the built-in systems. Here's a quick example with the Lotka-Volterra (predator-prey) system.

In [ ]:
lv_system = create_system("lotka_volterra", stochastic=True)

lv_config = Config(
    system_name="lotka_volterra",
    system_stochastic=True,
    system_num_timepoints=100,
    system_t0=0.0,
    system_t1=2.0,
    context_length=50,
    data_n_train_test_samples=512,
    data_batch_size=64,
    steps_per_epoch=100,
)

lv_train, lv_test, lv_ts = prepare_trajectories(lv_system, lv_config, jrandom.PRNGKey(1))
(lv_afm_train, lv_afm_test), _ = create_infinite_dataloaders(lv_train, lv_test, lv_ts, lv_config)

lv_model = AutoregressiveFlowMatching(
    dim_x=lv_system.data_dim,
    dim_c=1, dim_h=64, dim_t=16, dim_hidden=64,
    hidden_blocks=3, n_rnn_layers=3, rnn_hidden=64,
    rngs=nnx.Rngs(0),
)

lv_trainer = AFMTrainer(lv_model, learning_rate=1e-3)
lv_trainer.train(lv_afm_train, lv_afm_test, num_epochs=50, use_wandb=False)

# Predict
lv_history = lv_test[0, :50]
lv_cov = lv_ts[:50, None]

lv_samples = lv_model.sample(
    jrandom.PRNGKey(42), lv_history[None], lv_cov[None],
    nfe=20, n_steps_pred=30, n_samples=50,
)

# Plot
lv_samples_np = np.array(lv_samples[:, 0])
lv_mean = lv_samples_np.mean(0)
lv_lo = np.percentile(lv_samples_np, 2.5, axis=0)
lv_hi = np.percentile(lv_samples_np, 97.5, axis=0)
lv_gt = np.array(lv_test[0])
lv_ts_np = np.array(lv_ts)

fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
for d, name in enumerate(["Prey", "Predator"]):
    ax = axes[d]
    ax.plot(lv_ts_np[:50], np.array(lv_history[:, d]), "k-", label="History")
    pred_ts = lv_ts_np[50:80]
    ax.plot(pred_ts, lv_gt[50:80, d], "k--", alpha=0.5, label="Ground truth")
    ax.plot(pred_ts, lv_mean[:, d], "C1-", label="AFM mean")
    ax.fill_between(pred_ts, lv_lo[:, d], lv_hi[:, d], color="C1", alpha=0.2, label="95% CI")
    ax.set_ylabel(name)
    ax.grid(True, alpha=0.3)
    if d == 0:
        ax.legend(fontsize=8)

axes[-1].set_xlabel("Time")
axes[0].set_title("AFM Probabilistic Forecast — Lotka-Volterra")
plt.tight_layout()
plt.show()